# 06 — Risco de violação de OLA

**Objetivo:** estimar, no momento da abertura, a probabilidade de um incidente elegível a KPI estourar o prazo de OLA da sua prioridade.

**Algoritmo vencedor no último treino da plataforma:** Regressão logística (LogisticRegression, calibrada) · **métrica principal:** PR-AUC

Este notebook contém o treino completo do modelo, célula por célula: divisão temporal, candidatos,
competição na validação, reajuste, avaliação no teste e explicabilidade. O nome do arquivo reflete
o vencedor registrado na plataforma, mas **a escolha não está fixada no código**: todos os
candidatos competem de novo a cada execução, e vence quem tiver a melhor métrica na validação.

> Pré-requisito: executar antes o `02_engenharia_de_features.ipynb`, que gera `dados/base.pkl`.

In [ ]:
import os
import sqlite3
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
pd.set_option('display.width', 160)
pd.set_option('display.max_columns', 40)
plt.rcParams['figure.figsize'] = (11, 4)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25

PASTA = Path.cwd()
DADOS = PASTA / 'dados'            # saída do notebook 02, lida pelos notebooks de modelo
ARTEFATOS = PASTA / 'artefatos'    # modelos treinados por estes notebooks
DADOS.mkdir(exist_ok=True)
ARTEFATOS.mkdir(exist_ok=True)

SEMENTE = 42

base = pd.read_pickle(DADOS / 'base.pkl')
print('Base carregada:', f"{len(base['incidentes']):,} incidentes |",
      f"regime a partir de {base['inicio_regime']:%d/%m/%Y}")

## 1. Dados e divisão temporal

A matriz por incidente vem do notebook 02. A classe é rara (~1%), então as janelas de validação e
teste são maiores (45 dias cada) para conter positivos suficientes.

In [ ]:
X, y, identificacao = base['ola']['X'], base['ola']['y'], base['ola']['identificacao']
datas = identificacao['data']

DIAS_TESTE, DIAS_VALIDACAO = 45, 45
fim = datas.max()
inicio_teste = fim - pd.Timedelta(days=DIAS_TESTE - 1)
inicio_validacao = inicio_teste - pd.Timedelta(days=DIAS_VALIDACAO)
treino = datas < inicio_validacao
validacao = (datas >= inicio_validacao) & (datas < inicio_teste)
teste = datas >= inicio_teste

# Colunas constantes no treino não informam nada e sujam a importância.
uteis = [c for c in X.columns if X.loc[treino, c].nunique(dropna=False) > 1]
X_treino, X_validacao, X_teste = X.loc[treino, uteis], X.loc[validacao, uteis], X.loc[teste, uteis]
y_treino, y_validacao, y_teste = y[treino], y[validacao], y[teste]

pd.DataFrame({
    nome: {'linhas': int(m.sum()), 'início': datas[m].min().date(), 'fim': datas[m].max().date(),
           'violações': int(y[m].sum()), 'taxa': round(float(y[m].mean()), 4)}
    for nome, m in (('treino', treino), ('validação', validacao), ('teste', teste))
}).T

## 2. Métricas

In [ ]:
from sklearn.metrics import (
    average_precision_score, brier_score_loss, confusion_matrix, f1_score,
    precision_score, recall_score, roc_auc_score,
)


def metricas_classificacao(y_real, probabilidade, limiar=0.5):
    """Precisão, recall, F1, Brier, ROC-AUC e PR-AUC. Com classe rara, PR-AUC é a métrica honesta."""
    y = np.asarray(y_real).astype(int)
    prob = np.asarray(probabilidade, dtype=float)
    previsto = (prob >= limiar).astype(int)
    resultado = {
        'accuracy': float((previsto == y).mean()),
        'precision': float(precision_score(y, previsto, zero_division=0)),
        'recall': float(recall_score(y, previsto, zero_division=0)),
        'f1': float(f1_score(y, previsto, zero_division=0)),
        'brier': float(brier_score_loss(y, prob)),
        'limiar': float(limiar),
        'taxa_base': float(y.mean()),
        'positivos': int(y.sum()),
        'n': int(len(y)),
    }
    if 0 < y.sum() < len(y):
        resultado['roc_auc'] = float(roc_auc_score(y, prob))
        resultado['pr_auc'] = float(average_precision_score(y, prob))
    else:
        resultado['roc_auc'] = resultado['pr_auc'] = None
    vn, fp, fn, vp = confusion_matrix(y, previsto, labels=[0, 1]).ravel()
    resultado.update({'vn': int(vn), 'fp': int(fp), 'fn': int(fn), 'vp': int(vp)})
    return resultado


def melhor_limiar(y_real, probabilidade, passos=200):
    """Limiar que maximiza o F1 na VALIDAÇÃO. Com ~1% de positivos o corte 0,5 nunca dispararia."""
    y = np.asarray(y_real).astype(int)
    prob = np.asarray(probabilidade, dtype=float)
    if not y.sum():
        return 0.5
    candidatos = np.linspace(prob.min(), prob.max(), passos)[1:-1]
    if not len(candidatos):
        return 0.5
    pontuacoes = [f1_score(y, (prob >= c).astype(int), zero_division=0) for c in candidatos]
    return float(candidatos[int(np.argmax(pontuacoes))])

## 3. Candidatos (com peso para a classe positiva)

In [ ]:
from lightgbm import LGBMClassifier
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier


class RegraProbabilidade(BaseEstimator, ClassifierMixin):
    """Baseline: usa uma taxa histórica já calculada como probabilidade (a "regra fixa" sem ML)."""

    def __init__(self, coluna='taxa_violacao_media_28'):
        self.coluna = coluna

    def fit(self, X, y):
        self.indice_ = list(X.columns).index(self.coluna)
        self.classes_ = np.array([0, 1])
        return self

    def predict_proba(self, X):
        p = np.clip(np.asarray(X.iloc[:, self.indice_], dtype=float), 0.0, 1.0)
        return np.column_stack([1 - p, p])

    def predict(self, X):
        return (self.predict_proba(X)[:, 1] >= 0.5).astype(int)


def peso_para_desbalanceamento(y):
    """scale_pos_weight = negativos / positivos."""
    positivos = float(np.sum(y))
    return (len(y) - positivos) / positivos if positivos else 1.0


def candidatos_classificacao(peso_positivo, coluna_regra):
    """Classe desbalanceada: todos os modelos treinam com peso na classe positiva."""
    return {
        'Baseline (taxa histórica)': {
            'tipo': 'baseline',
            'estimador': lambda: RegraProbabilidade(coluna=coluna_regra),
            'grade': [{}],
        },
        'Baseline (prevalência)': {
            'tipo': 'baseline',
            'estimador': lambda: DummyClassifier(strategy='prior', random_state=SEMENTE),
            'grade': [{}],
        },
        'Regressão logística': {
            'tipo': 'linear',
            'estimador': lambda: Pipeline([
                ('escala', StandardScaler()),
                ('modelo', LogisticRegression(max_iter=2000, class_weight='balanced', random_state=SEMENTE)),
            ]),
            'grade': [{'modelo__C': c} for c in (0.05, 0.5, 5.0)],
        },
        'Random Forest': {
            'tipo': 'arvore',
            'estimador': lambda: RandomForestClassifier(random_state=SEMENTE, n_jobs=-1,
                                                        class_weight='balanced_subsample'),
            'grade': [
                {'n_estimators': 300, 'min_samples_leaf': 20, 'max_features': 0.5, 'max_depth': 10},
                {'n_estimators': 300, 'min_samples_leaf': 20, 'max_features': 0.5, 'max_depth': 14},
                {'n_estimators': 300, 'min_samples_leaf': 10, 'max_features': 0.8, 'max_depth': 8},
            ],
        },
        'XGBoost': {
            'tipo': 'arvore',
            'estimador': lambda: XGBClassifier(random_state=SEMENTE, n_jobs=-1, tree_method='hist',
                                               objective='binary:logistic', eval_metric='aucpr',
                                               scale_pos_weight=peso_positivo),
            'grade': [
                {'n_estimators': 400, 'learning_rate': 0.05, 'max_depth': 4,
                 'subsample': 0.8, 'colsample_bytree': 0.8, 'reg_lambda': 2.0},
                {'n_estimators': 250, 'learning_rate': 0.08, 'max_depth': 3,
                 'subsample': 0.9, 'colsample_bytree': 0.7, 'reg_lambda': 5.0},
            ],
        },
        'LightGBM': {
            'tipo': 'arvore',
            'estimador': lambda: LGBMClassifier(random_state=SEMENTE, n_jobs=-1, verbose=-1,
                                                class_weight='balanced'),
            'grade': [
                {'n_estimators': 400, 'learning_rate': 0.05, 'num_leaves': 15,
                 'min_child_samples': 30, 'colsample_bytree': 0.8},
                {'n_estimators': 250, 'learning_rate': 0.08, 'num_leaves': 7, 'min_child_samples': 60},
            ],
        },
    }

In [ ]:
PESO_POSITIVO = peso_para_desbalanceamento(y_treino)
CANDIDATOS = candidatos_classificacao(peso_positivo=PESO_POSITIVO, coluna_regra='taxa_violacao_equipe')
print(f'Peso da classe positiva (negativos/positivos): {PESO_POSITIVO:.1f}')

## 4. Competição na validação (métrica: PR-AUC)

In [ ]:
import time


def instanciar(definicao, parametros):
    estimador = definicao['estimador']()
    return estimador.set_params(**parametros) if parametros else estimador


def competir(candidatos, X_treino, y_treino, X_validacao, y_validacao, avaliar, chave_metrica, maior_melhor=False):
    """Protocolo de seleção:
    1. cada configuração de cada algoritmo treina SÓ no treino;
    2. o vencedor é escolhido pela VALIDAÇÃO (o teste não participa da decisão);
    3. depois o vencedor é reajustado em treino + validação e medido UMA vez no teste.
    """
    tabela, melhor = [], None
    for nome, definicao in candidatos.items():
        for parametros in definicao['grade']:
            modelo = instanciar(definicao, parametros)
            inicio = time.perf_counter()
            modelo.fit(X_treino, y_treino)
            segundos = time.perf_counter() - inicio
            resultado = avaliar(modelo, X_validacao, y_validacao)
            tabela.append({'candidato': nome, 'tipo': definicao['tipo'], 'hiperparametros': parametros,
                           'segundos_treino': round(segundos, 1), **resultado})
            pontuacao = resultado.get(chave_metrica)
            if pontuacao is None:
                continue
            if melhor is None or (pontuacao > melhor['pontuacao'] if maior_melhor else pontuacao < melhor['pontuacao']):
                melhor = {'nome': nome, 'definicao': definicao, 'parametros': parametros,
                          'pontuacao': pontuacao, 'modelo': modelo}
    tabela = pd.DataFrame(tabela)
    tabela['vencedor'] = (tabela['candidato'] == melhor['nome']) & (
        tabela['hiperparametros'].astype(str) == str(melhor['parametros']))
    return melhor, tabela


def reajustar(melhor, X, y):
    """Mesma configuração vencedora, reajustada sobre treino + validação."""
    modelo = instanciar(melhor['definicao'], melhor['parametros'])
    modelo.fit(X, y)
    return modelo

In [ ]:
def avaliar_classificacao(modelo, X, y):
    return metricas_classificacao(y, modelo.predict_proba(X)[:, 1])


melhor, tabela = competir(CANDIDATOS, X_treino, y_treino, X_validacao, y_validacao,
                          avaliar=avaliar_classificacao, chave_metrica='pr_auc', maior_melhor=True)
print('Vencedor na validação:', melhor['nome'], melhor['parametros'])
tabela[['candidato', 'tipo', 'hiperparametros', 'pr_auc', 'roc_auc', 'brier', 'segundos_treino', 'vencedor']] \
    .sort_values('pr_auc', ascending=False).round(4)

## 5. Calibração de Platt, limiar e avaliação no teste

Treinar com `class_weight='balanced'` faz a probabilidade média prevista ficar muito acima da taxa
real. A calibração corrige a escala sem mudar a ordenação dos casos.

In [ ]:
from sklearn.calibration import CalibratedClassifierCV
from sklearn.frozen import FrozenEstimator


def calibrar(modelo_base, X_validacao, y_validacao, X_teste, y_teste):
    """Calibração de Platt (sigmoide) ajustada na validação, com o modelo base congelado.

    Treinar com peso de classe distorce a escala da saída: a probabilidade média prevista fica
    muito acima da taxa real. A calibração não muda a ordenação dos casos (ROC-AUC igual); muda a
    escala, para que o número possa ser lido como probabilidade.
    """
    calibrado = CalibratedClassifierCV(FrozenEstimator(modelo_base), method='sigmoid').fit(X_validacao, y_validacao)
    antes = modelo_base.predict_proba(X_teste)[:, 1]
    depois = calibrado.predict_proba(X_teste)[:, 1]
    diagnostico = pd.DataFrame({
        'antes da calibração': [brier_score_loss(y_teste, antes), antes.mean()],
        'depois da calibração': [brier_score_loss(y_teste, depois), depois.mean()],
        'taxa real no teste': [None, float(np.mean(y_teste))],
    }, index=['Brier', 'probabilidade média'])
    return calibrado, diagnostico

In [ ]:
modelo_base = melhor['modelo']
modelo, diagnostico_calibracao = calibrar(modelo_base, X_validacao, y_validacao, X_teste, y_teste)
display(diagnostico_calibracao.round(5))

limiar = melhor_limiar(y_validacao, modelo.predict_proba(X_validacao)[:, 1])
resultado_teste = metricas_classificacao(y_teste, modelo.predict_proba(X_teste)[:, 1], limiar)
print(f'Limiar escolhido na validação: {limiar:.4f}')
pd.Series(resultado_teste, name='teste (model_ola)').round(4)

In [ ]:
from sklearn.metrics import precision_recall_curve, roc_curve

prob_teste = modelo.predict_proba(X_teste)[:, 1]
precisao, revocacao, _ = precision_recall_curve(y_teste, prob_teste)
fpr, tpr, _ = roc_curve(y_teste, prob_teste)
fig, (a, b) = plt.subplots(1, 2, figsize=(11, 4))
a.plot(revocacao, precisao, color='#2563eb'); a.axhline(y_teste.mean(), ls='--', color='#94a3b8', label='taxa-base')
a.set_xlabel('recall'); a.set_ylabel('precisão'); a.set_title('Curva precisão-recall (teste)'); a.legend()
b.plot(fpr, tpr, color='#2563eb'); b.plot([0, 1], [0, 1], ls='--', color='#94a3b8')
b.set_xlabel('falso positivo'); b.set_ylabel('verdadeiro positivo'); b.set_title('Curva ROC (teste)')
plt.tight_layout(); plt.show()

## 6. Explicabilidade

In [ ]:
from sklearn.inspection import permutation_importance


def estimador_final(modelo):
    """Desembrulha Pipeline e calibrador até o estimador que carrega os pesos."""
    if isinstance(modelo, Pipeline):
        return estimador_final(modelo.steps[-1][1])
    if type(modelo).__name__ in ('CalibratedClassifierCV', 'FrozenEstimator'):
        interno = getattr(modelo, 'estimator', None)
        if interno is not None:
            return estimador_final(interno)
    return modelo


def importancia_por_permutacao(modelo, X, y, scoring, repeticoes=5, limite=20):
    """Quanto a métrica piora ao embaralhar cada feature — medido na validação, fora do treino."""
    resultado = permutation_importance(modelo, X, y, scoring=scoring, n_repeats=repeticoes,
                                       random_state=SEMENTE, n_jobs=1)
    return (pd.DataFrame({'feature': X.columns, 'queda_metrica': resultado.importances_mean,
                          'desvio': resultado.importances_std})
            .sort_values('queda_metrica', ascending=False).head(limite).reset_index(drop=True))

In [ ]:
importancia_por_permutacao(modelo, X_validacao, y_validacao, scoring='average_precision').round(5)

### Contribuição por feature (SHAP)

Para modelo linear, o valor de Shapley tem forma fechada: `coeficiente × (valor padronizado − média
do treino)`. Para árvore, usa-se o `TreeExplainer`. Abaixo, o SHAP médio dos incidentes dos últimos
28 dias por equipe — o que empurra o risco de cada equipe para cima ou para baixo.

In [ ]:
class ExplicadorLinear:
    def __init__(self, modelo, X_referencia):
        self._preparo = modelo[:-1] if isinstance(modelo, Pipeline) else None
        estimador = estimador_final(modelo)
        self.coeficientes = np.ravel(estimador.coef_).astype(float)
        self.media = self._transformar(X_referencia).mean(axis=0)

    def _transformar(self, X):
        return np.asarray(X if self._preparo is None else self._preparo.transform(X), dtype=float)

    def shap_values(self, X):
        return (self._transformar(X) - self.media) * self.coeficientes


estimador = estimador_final(modelo_base)
if hasattr(estimador, 'coef_'):
    explicador = ExplicadorLinear(modelo_base, X_treino)
else:
    import shap
    explicador = shap.TreeExplainer(estimador)

recentes = (identificacao['data'] >= fim - pd.Timedelta(days=28)).to_numpy()
X_recentes = X.loc[recentes, uteis]
valores = np.asarray(explicador.shap_values(X_recentes))
if valores.ndim == 3:
    valores = valores[..., -1]
shap_por_equipe = pd.DataFrame(valores, columns=uteis).groupby(identificacao.loc[recentes, 'equipe'].to_numpy()).mean()

perfil = identificacao.loc[recentes].assign(probabilidade=modelo.predict_proba(X_recentes)[:, 1])
resumo_equipes = perfil.groupby('equipe')['probabilidade'].agg(['size', 'mean']).sort_values('mean', ascending=False)
resumo_equipes.columns = ['incidentes (28d)', 'probabilidade média']
display(resumo_equipes.round(4))

for equipe in resumo_equipes.index[:3]:
    topo = shap_por_equipe.loc[equipe].reindex(shap_por_equipe.loc[equipe].abs().sort_values(ascending=False).index)[:5]
    print(f'\n{equipe}:')
    for feature, valor in topo.items():
        print(f'   {"aumenta" if valor > 0 else "reduz  "} {feature:<34} {valor:+.4f}')

## 7. Artefato

In [ ]:
import joblib

joblib.dump({'modelo': modelo, 'colunas': uteis, 'limiar': limiar, 'explicador': explicador,
             'vencedor': melhor['nome'], 'hiperparametros': melhor['parametros'], 'teste': resultado_teste},
            ARTEFATOS / 'model_ola.joblib', compress=3)
print('Artefato gravado em', ARTEFATOS / 'model_ola.joblib')